In [1]:
import torch
import numpy as np
from torch import nn
from torch.nn import functional as F
from torch import optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from pathlib import Path
import os
from PIL import Image
import time
import random
from collections import defaultdict
from torch.nn.functional import cosine_similarity
from tqdm import tqdm
from torchvision.transforms import functional as TF
from torch.utils.data import Sampler
import random
from collections import defaultdict
import math

In [2]:
# custom vibed dataloader with index file
class IndexedDataset(Dataset):
    def __init__(self, index_file, transform=None):
        self.index_file = index_file
        self.transform = transform
        self.data = []
        self.labels = []
        self.load_data()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path = self.data[idx]
        label = self.labels[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

    def load_data(self):
        raw_labels = []
        with open(self.index_file, "r") as f:
            for line in f:
                path, label = line.strip().split()
                self.data.append(path)
                raw_labels.append(int(label))
        # Map to contiguous 0..num_classes-1 so ArcFace head indices are valid
        unique = sorted(set(raw_labels))
        self._label_to_idx = {y: i for i, y in enumerate(unique)}
        self.labels = [self._label_to_idx[y] for y in raw_labels]

In [3]:


# p - identities, k - images per identity
# data sampler
class PKSampler(Sampler):
    def __init__(self, labels, P=64, K=8):
        self.P = P
        self.K = K
        self.labels = labels
        self.label_to_indices = defaultdict(list)
        for idx, y in enumerate(labels):
            self.label_to_indices[y].append(idx)
        self.labels_unique = list(self.label_to_indices.keys())

    def __iter__(self):
        random.shuffle(self.labels_unique)
        batch = []
        for label in self.labels_unique:
            candidates = self.label_to_indices[label]
            if len(candidates) >= self.K:
                indices = random.sample(candidates, k=self.K)
            else:
                # sample with replacement if class is too small
                indices = random.choices(candidates, k=self.K)
            batch.extend(indices)
            if len(batch) == self.P * self.K:
                yield from batch
                batch = []

    def __len__(self):
        return (len(self.labels_unique) // self.P) * self.P * self.K

In [4]:
def read_lfw_pairs(pair_file, root_dir):
    pairs = []
    with open(pair_file, "r") as f:
        for line in f:
            if line.strip() == "" or line.startswith("#"):
                continue
            parts = line.strip().split()

            # Format A: "img1.jpg img2.jpg label" (all files in root_dir)
            if len(parts) == 3 and parts[0].endswith(".jpg") and parts[1].endswith(".jpg"):
                p1, p2, label = parts
                p1 = Path(root_dir) / p1
                p2 = Path(root_dir) / p2
                pairs.append((str(p1), str(p2), int(label)))
                continue

            # Format B: LFW standard name/index
            if len(parts) == 3:
                name, i1, i2 = parts
                f1 = i1 if i1.endswith(".jpg") else f"{int(i1):04d}.jpg"
                f2 = i2 if i2.endswith(".jpg") else f"{int(i2):04d}.jpg"
                p1 = Path(root_dir) / name / f"{name}_{f1}"
                p2 = Path(root_dir) / name / f"{name}_{f2}"
                pairs.append((str(p1), str(p2), 1))
            elif len(parts) == 4:
                name1, i1, name2, i2 = parts
                f1 = i1 if i1.endswith(".jpg") else f"{int(i1):04d}.jpg"
                f2 = i2 if i2.endswith(".jpg") else f"{int(i2):04d}.jpg"
                p1 = Path(root_dir) / name1 / f"{name1}_{f1}"
                p2 = Path(root_dir) / name2 / f"{name2}_{f2}"
                pairs.append((str(p1), str(p2), 0))
    return pairs

def read_pairs_from_file(pair_file, root_dir=None):
    pairs = []
    root_dir = Path(root_dir) if root_dir is not None else None
    with open(pair_file, "r") as f:
        for line in f:
            if line.strip() == "" or line.startswith("#"):
                continue
            p1, p2, label = line.strip().split()
            if root_dir is not None:
                if not os.path.isabs(p1):
                    p1 = root_dir / p1
                if not os.path.isabs(p2):
                    p2 = root_dir / p2
            pairs.append((str(p1), str(p2), int(label)))
    return pairs

In [ ]:
data_dir = 'data'


transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])


data_root = Path("~/Datasets").expanduser()

train_dataset = IndexedDataset(
    data_root / "ms1m-arcface" / "index.txt",
    transform=transform,
)

# --- MobileFaceNet-style eval datasets/loaders ---
import sys
mfn_repo = Path("/home/xerneas/Coding/MobileFaceNet_Tutorial_Pytorch")
sys.path.append(str(mfn_repo))
from data_set.dataloader import LFW as MF_LFW, CFP_FP as MF_CFP_FP, AgeDB30 as MF_AgeDB30

# MobileFaceNet eval transform (no resize)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

lfw_dataset = MF_LFW(
    root=str(data_root / "LFW" / "lfw_align_112"),
    file_list=str(data_root / "LFW" / "pairs.txt"),
    transform=eval_transform,
)
cfp_dataset = MF_CFP_FP(
    root=str(data_root / "CFP-FP" / "CFP_FP_aligned_112"),
    file_list=str(data_root / "CFP-FP" / "cfp_fp_pair.txt"),
    transform=eval_transform,
)
agedb_dataset = MF_AgeDB30(
    root=str(data_root / "AgeDB-30" / "agedb30_align_112"),
    file_list=str(data_root / "AgeDB-30" / "agedb_30_pair.txt"),
    transform=eval_transform,
)

eval_loaders = {
    "LFW": DataLoader(lfw_dataset, batch_size=128, shuffle=False, num_workers=2, drop_last=False),
    "CFP-FP": DataLoader(cfp_dataset, batch_size=128, shuffle=False, num_workers=2, drop_last=False),
    "AgeDB-30": DataLoader(agedb_dataset, batch_size=128, shuffle=False, num_workers=2, drop_last=False),
}

pairs_lfw = read_lfw_pairs(
    pair_file=data_root / "LFW" / "pairs.txt",
    root_dir=data_root / "LFW" / "lfw_align_112",
)

pairs_cfp = read_pairs_from_file(
    data_root / "CFP-FP" / "cfp_fp_pair.txt",
    root_dir=data_root / "CFP-FP" / "CFP_FP_aligned_112",
)

pairs_agedb = read_pairs_from_file(
    data_root / "AgeDB-30" / "agedb_30_pair.txt",
    root_dir=data_root / "AgeDB-30" / "agedb30_align_112",
)

# Plain shuffle like MobileFaceNet_Tutorial_Pytorch: use all images every epoch
train_loader = DataLoader(
    train_dataset,
    batch_size=192,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [6]:
emb_dim = 512
num_epochs = 20
global_step = 0
P = 64
K = 8

ckpt_dir = Path("checkpoints")
ckpt_dir.mkdir(exist_ok=True)
num_classes = len(set(train_dataset.labels))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_ids = len(set(train_dataset.labels))
num_samples = len(train_dataset)
print("num identities:", num_ids)
print("num samples:", num_samples)
print("train_loader: shuffle=True (no PK sampler)")

num identities: 85742
num samples: 5822653
train_loader: shuffle=True (no PK sampler)


In [7]:
print("train size:", len(train_dataset))

print("train sample 0:", train_dataset.data[0], train_dataset.labels[0])

for i in range(5):
    print("train", i, train_dataset.data[i], train_dataset.labels[i])

train size: 5822653
train sample 0: /home/xerneas/Datasets/ms1m-arcface/0/37.jpg 0
train 0 /home/xerneas/Datasets/ms1m-arcface/0/37.jpg 0
train 1 /home/xerneas/Datasets/ms1m-arcface/0/8.jpg 0
train 2 /home/xerneas/Datasets/ms1m-arcface/0/66.jpg 0
train 3 /home/xerneas/Datasets/ms1m-arcface/0/27.jpg 0
train 4 /home/xerneas/Datasets/ms1m-arcface/0/22.jpg 0


In [ ]:
class MiniCNN(nn.Module):
    def __init__(self, emb_dim=256):
        super(MiniCNN, self).__init__()
        self.emb_dim = emb_dim

        self.encoder = nn.Sequential(
            # mix rgb channels
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # depthwise 1
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, groups=32),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # pointwise 1
            nn.Conv2d(32, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # depthwise 2
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, groups=64),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # pointwise 2
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU()            
            
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, emb_dim)

    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x).flatten(1)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)
        return x
    

        

In [9]:
# ArcFace head (exact MobileFaceNet implementation)
def l2_norm(input, axis=1):
    norm = torch.norm(input, 2, axis, True)
    output = torch.div(input, norm)
    return output


class Arcface(nn.Module):
    # implementation of additive margin softmax loss in https://arxiv.org/abs/1801.05599
    def __init__(self, embedding_size=512, classnum=51332, s=64., m=0.5):
        super(Arcface, self).__init__()
        self.classnum = classnum
        self.kernel = nn.Parameter(torch.Tensor(embedding_size, classnum))
        nn.init.xavier_uniform_(self.kernel)
        # initial kernel
        self.kernel.data.uniform_(-1, 1).renorm_(2, 1, 1e-5).mul_(1e5)
        self.m = m
        self.s = s
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.mm = self.sin_m * m  # issue 1
        self.threshold = math.cos(math.pi - m)

    def forward(self, embbedings, label):
        # weights norm
        nB = len(embbedings)
        kernel_norm = l2_norm(self.kernel, axis=0)
        # cos(theta+m)
        cos_theta = torch.mm(embbedings, kernel_norm)
        cos_theta = cos_theta.clamp(-1, 1)  # for numerical stability
        cos_theta_2 = torch.pow(cos_theta, 2)
        sin_theta_2 = 1 - cos_theta_2
        sin_theta = torch.sqrt(sin_theta_2)
        cos_theta_m = (cos_theta * self.cos_m - sin_theta * self.sin_m)
        # this condition controls the theta+m should in range [0, pi]
        cond_v = cos_theta - self.threshold
        cond_mask = cond_v <= 0
        keep_val = (cos_theta - self.mm)  # when theta not in [0,pi], use cosface instead
        cos_theta_m[cond_mask] = keep_val[cond_mask]
        output = cos_theta * 1.0  # prevent in_place operation on cos_theta
        idx_ = torch.arange(0, nB, dtype=torch.long, device=embbedings.device)
        output[idx_, label] = cos_theta_m[idx_, label]
        output *= self.s
        return output

In [10]:
def build_pairs_from_index(index_file, num_pairs=2000):
    # expects: "path label"
    from collections import defaultdict
    import random

    label_to_paths = defaultdict(list)
    with open(index_file, "r") as f:
        for line in f:
            path, label = line.strip().split()
            label_to_paths[int(label)].append(path)

    labels = list(label_to_paths.keys())
    pairs = []

    # same-person pairs
    while len(pairs) < num_pairs // 2:
        label = random.choice(labels)
        if len(label_to_paths[label]) < 2:
            continue
        p1, p2 = random.sample(label_to_paths[label], 2)
        pairs.append((p1, p2, 1))

    # different-person pairs
    while len(pairs) < num_pairs:
        l1, l2 = random.sample(labels, 2)
        p1 = random.choice(label_to_paths[l1])
        p2 = random.choice(label_to_paths[l2])
        pairs.append((p1, p2, 0))

    random.shuffle(pairs)
    return pairs

In [11]:
from torch.nn.functional import cosine_similarity
#embeddings comparison
def verify(model, transform, pairs, device, threshold=0.5):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for p1, p2, y in pairs:
            img1 = transform(Image.open(p1).convert("RGB")).unsqueeze(0).to(device)
            img2 = transform(Image.open(p2).convert("RGB")).unsqueeze(0).to(device)

            emb1 = model(img1)
            emb2 = model(img2)

            sim = cosine_similarity(emb1, emb2).item()
            pred = 1 if sim >= threshold else 0

            correct += (pred == y)
            total += 1

    return correct / max(total, 1)

In [12]:
model = MiniCNN(emb_dim).to(device)
head = Arcface(embedding_size=emb_dim, classnum=num_classes, s=64., m=0.5).to(device)

# Lower LR: 0.1 was too high (train_acc stayed 0). 0.01–0.02 works better for 85k-class ArcFace.
optimizer = torch.optim.SGD(
    list(model.parameters()) + list(head.parameters()),
    lr=0.01,
    momentum=0.9,
    nesterov=True,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[6, 10, 14],
    gamma=0.3
)



In [13]:
class PathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return path, img


@torch.no_grad()
def embed_image(model, img_pil, transform, device, flip=False):
    # original
    img = transform(img_pil).unsqueeze(0).to(device)
    emb = model(img)

    if flip:
        img_f = transform(TF.hflip(img_pil)).unsqueeze(0).to(device)
        emb_f = model(img_f)
        emb = (emb + emb_f) / 2.0

    # ensure normalized embeddings
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb

@torch.no_grad()
def compute_embeddings(model, paths, transform, device, flip=False, batch_size=256, num_workers=4):
    dataset = PathDataset(paths, transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    model.eval()
    embs = {}

    for batch_paths, imgs in tqdm(loader, desc="embed", leave=False):
        imgs = imgs.to(device)
        emb = model(imgs)

        if flip:
            imgs_f = torch.flip(imgs, dims=[3])
            emb_f = model(imgs_f)
            emb = (emb + emb_f) / 2.0

        emb = torch.nn.functional.normalize(emb, p=2, dim=1).cpu()
        for path, vec in zip(batch_paths, emb):
            embs[path] = vec

    return embs

def l2_norm(input, axis=1):
    norm = torch.norm(input, 2, axis, True)
    return input / (norm + 1e-12)


def getAccuracy(scores, flags, threshold, method):
    if method == "l2_distance":
        p = np.sum(scores[flags == 1] < threshold)
        n = np.sum(scores[flags == -1] > threshold)
    elif method == "cos_distance":
        p = np.sum(scores[flags == 1] > threshold)
        n = np.sum(scores[flags == -1] < threshold)
    return 1.0 * (p + n) / len(scores)


def getThreshold(scores, flags, thrNum, method):
    accuracys = np.zeros((2 * thrNum + 1, 1))
    thresholds = np.arange(-thrNum, thrNum + 1) * 3.0 / thrNum
    for i in range(2 * thrNum + 1):
        accuracys[i] = getAccuracy(scores, flags, thresholds[i], method)
    max_index = np.squeeze(accuracys == np.max(accuracys))
    bestThreshold = np.mean(thresholds[max_index])
    return bestThreshold


def getFeature_mfn(net, dataloader, device, flip=True):
    featureLs = None
    featureRs = None

    for det in dataloader:
        for i in range(len(det)):
            det[i] = det[i].to(device)

        with torch.no_grad():
            res = [net(d).data.cpu() for d in det]

        if flip:
            featureL = l2_norm(res[0] + res[1])
            featureR = l2_norm(res[2] + res[3])
        else:
            featureL = res[0]
            featureR = res[2]

        if featureLs is None:
            featureLs = featureL
        else:
            featureLs = torch.cat((featureLs, featureL), 0)
        if featureRs is None:
            featureRs = featureR
        else:
            featureRs = torch.cat((featureRs, featureR), 0)

    return featureLs, featureRs


def evaluation_10_fold_mfn(featureL, featureR, dataset, method="l2_distance"):
    ACCs = np.zeros(10)
    threshold = np.zeros(10)
    fold = np.array(dataset.folds).reshape(1, -1)
    flags = np.array(dataset.flags).reshape(1, -1)
    flags_1d = np.squeeze(flags)

    featureL_np = featureL.numpy() if hasattr(featureL, "numpy") else np.asarray(featureL)
    featureR_np = featureR.numpy() if hasattr(featureR, "numpy") else np.asarray(featureR)

    for i in range(10):
        valFold = (fold != i).ravel()
        testFold = (fold == i).ravel()

        featureLs = featureL_np.copy()
        featureRs = featureR_np.copy()

        mu = np.mean(np.concatenate((featureLs[valFold, :], featureRs[valFold, :]), 0), 0)
        mu = np.expand_dims(mu, 0)
        featureLs = featureLs - mu
        featureRs = featureRs - mu
        featureLs = featureLs / np.expand_dims(np.sqrt(np.sum(np.power(featureLs, 2), 1)), 1)
        featureRs = featureRs / np.expand_dims(np.sqrt(np.sum(np.power(featureRs, 2), 1)), 1)

        if method == "l2_distance":
            scores = np.sum(np.power((featureLs - featureRs), 2), 1)
        elif method == "cos_distance":
            scores = np.sum(np.multiply(featureLs, featureRs), 1)

        threshold[i] = getThreshold(scores[valFold], flags_1d[valFold], 10000, method)
        ACCs[i] = getAccuracy(scores[testFold], flags_1d[testFold], threshold[i], method)

    return ACCs, threshold


# Legacy verify_10fold retained for optional use


In [14]:
lfw_featL, lfw_featR = getFeature_mfn(model, eval_loaders["LFW"], device, flip=True)
lfw_accs, lfw_thr = evaluation_10_fold_mfn(lfw_featL, lfw_featR, lfw_dataset, method="l2_distance")
print("LFW average acc: {:.4f} average threshold: {:.4f}".format(np.mean(lfw_accs) * 100, np.mean(lfw_thr)))

cfp_featL, cfp_featR = getFeature_mfn(model, eval_loaders["CFP-FP"], device, flip=True)
cfp_accs, cfp_thr = evaluation_10_fold_mfn(cfp_featL, cfp_featR, cfp_dataset, method="l2_distance")
print("CFP-FP average acc: {:.4f} average threshold: {:.4f}".format(np.mean(cfp_accs) * 100, np.mean(cfp_thr)))

agedb_featL, agedb_featR = getFeature_mfn(model, eval_loaders["AgeDB-30"], device, flip=True)
agedb_accs, agedb_thr = evaluation_10_fold_mfn(agedb_featL, agedb_featR, agedb_dataset, method="l2_distance")
print("AgeDB-30 average acc: {:.4f} average threshold: {:.4f}".format(np.mean(agedb_accs) * 100, np.mean(agedb_thr)))

LFW average acc: 62.2833 average threshold: 1.4740
CFP-FP average acc: 60.3000 average threshold: 1.7927
AgeDB-30 average acc: 52.2333 average threshold: 2.0797


In [15]:
# --- overfit sanity check (optional) ---
# Set RUN_OVERFIT = True to run; then restart kernel before full training.
RUN_OVERFIT = False

if RUN_OVERFIT:
    import random
    from collections import defaultdict

    # pick a few identities and a few images per identity
    id_to_indices = defaultdict(list)
    for idx, y in enumerate(train_dataset.labels):
        id_to_indices[y].append(idx)

    random.seed(123)
    small_ids = random.sample(list(id_to_indices.keys()), 10)
    small_indices = []
    for y in small_ids:
        small_indices += id_to_indices[y][:8]  # 8 images per identity

    # remap labels to 0..(N-1) for the tiny subset
    id_map = {y: i for i, y in enumerate(small_ids)}

    class RemappedSubset(Dataset):
        def __init__(self, base, indices, id_map):
            self.base = base
            self.indices = list(indices)
            self.id_map = id_map

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, idx):
            img, label = self.base[self.indices[idx]]
            return img, self.id_map[label]

    small_dataset = RemappedSubset(train_dataset, small_indices, id_map)
    small_loader = DataLoader(
        small_dataset,
        batch_size=4,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

    print("overfit small_ids:", small_ids)
    print("mapped labels sample:", [small_dataset[i][1] for i in range(min(10, len(small_dataset)))])

    # fresh model + simple linear head for sanity check
    overfit_model = MiniCNN(emb_dim).to(device)
    overfit_head = nn.Linear(emb_dim, len(small_ids)).to(device)
    overfit_opt = torch.optim.SGD(
        list(overfit_model.parameters()) + list(overfit_head.parameters()),
        lr=0.02,
        momentum=0.9,
        nesterov=True,
        weight_decay=0.0,
    )

    for epoch in range(50):
        overfit_model.train()
        overfit_head.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for imgs, labels in small_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            overfit_opt.zero_grad()
            # bypass embedding normalization for sanity check
            emb = overfit_model.encoder(imgs)
            emb = overfit_model.pool(emb).flatten(1)
            emb = overfit_model.fc(emb)
            logits = overfit_head(emb)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            overfit_opt.step()

            running_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        print(
            f"[overfit] epoch {epoch+1} loss={running_loss/max(1,len(small_loader)):.4f} "
            f"acc={correct/max(1,total):.4f}"
        )

    # --- embedding sanity checks (optional) ---
    RUN_EMB_TEST = True

    if RUN_EMB_TEST:
        import numpy as np

        def sample_pairs_from_ids(dataset, ids, per_id=5, max_pairs=200):
            id_to_paths = {}
            for p, y in zip(dataset.data, dataset.labels):
                if y in ids:
                    id_to_paths.setdefault(y, []).append(p)

            pairs = []
            for y in ids:
                paths = id_to_paths[y][:per_id]
                for i in range(len(paths) - 1):
                    pairs.append((paths[i], paths[i + 1], 1))

            ys = list(ids)
            for _ in range(max_pairs):
                y1, y2 = random.sample(ys, 2)
                p1 = random.choice(id_to_paths[y1])
                p2 = random.choice(id_to_paths[y2])
                pairs.append((p1, p2, 0))

            return pairs

        # evaluate embeddings from the overfit model on the same small dataset
        test_ids = small_ids
        test_pairs = sample_pairs_from_ids(train_dataset, test_ids)
        mean_acc, mean_t, _ = verify_10fold(
            overfit_model,
            transform,
            test_pairs,
            device,
            flip=False,
            thresholds=np.linspace(0, 4, 401),
            seed=123,
        )
        print("[emb] sampled pair acc (overfit):", mean_acc, "t:", mean_t)

        def knn_sanity(model, dataset, n=80, k=3):
            idxs = np.random.choice(len(dataset), n, replace=False)
            imgs = torch.stack([dataset[i][0] for i in idxs]).to(device)
            labels = torch.tensor([dataset[i][1] for i in idxs])

            with torch.no_grad():
                emb = model(imgs).cpu()
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)

            sims = emb @ emb.T
            np.fill_diagonal(sims.numpy(), -1)

            topk = sims.topk(k, dim=1).indices
            correct = 0
            for i in range(n):
                correct += (labels[topk[i]] == labels[i]).any().item()
            print("[emb] KNN hit@{} (overfit):".format(k), correct / n)

        knn_sanity(overfit_model, small_dataset, n=min(80, len(small_dataset)), k=3)


In [ ]:
# train loop
start_time = time.time()

verif_interval = 2000
last_verif_acc = None
last_verif_t = None
global_step = 0

for epoch in range(num_epochs):
    model.train()
    head.train()

    running_loss = 0.0
    correct = 0
    total = 0
    epoch_start = time.time()

    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{num_epochs}")
    for imgs, labels in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        embeddings = model(imgs)
        logits = head(embeddings, labels)
        loss = F.cross_entropy(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(head.parameters()), max_norm=1.0
        )
        optimizer.step()

        running_loss += loss.item()

        # Accuracy on raw cosine logits (more meaningful than margin logits)
        with torch.no_grad():
            emb_norm = F.normalize(embeddings, dim=1)
            W_norm = F.normalize(head.kernel, dim=0)
            cos_logits = emb_norm @ W_norm
            preds = torch.argmax(cos_logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        global_step += 1
        if global_step % verif_interval == 0:
            pbar.clear()
            pbar.disable = True
            pbar.write(f"Running verification at step {global_step}...")

            # LFW (MobileFaceNet-style)
            lfw_featL, lfw_featR = getFeature_mfn(model, eval_loaders["LFW"], device, flip=True)
            lfw_accs, lfw_thr = evaluation_10_fold_mfn(lfw_featL, lfw_featR, lfw_dataset, method="l2_distance")
            lfw_acc = float(np.mean(lfw_accs) * 100)
            lfw_t = float(np.mean(lfw_thr))
            pbar.write(f"Verification done (LFW): acc={lfw_acc:.2f}%, t={lfw_t:.4f}")

            # CFP-FP (MobileFaceNet-style)
            cfp_featL, cfp_featR = getFeature_mfn(model, eval_loaders["CFP-FP"], device, flip=True)
            cfp_accs, cfp_thr = evaluation_10_fold_mfn(cfp_featL, cfp_featR, cfp_dataset, method="l2_distance")
            cfp_acc = float(np.mean(cfp_accs) * 100)
            cfp_t = float(np.mean(cfp_thr))
            pbar.write(f"Verification done (CFP-FP): acc={cfp_acc:.2f}%, t={cfp_t:.4f}")

            # AgeDB-30 (MobileFaceNet-style)
            agedb_featL, agedb_featR = getFeature_mfn(model, eval_loaders["AgeDB-30"], device, flip=True)
            agedb_accs, agedb_thr = evaluation_10_fold_mfn(agedb_featL, agedb_featR, agedb_dataset, method="l2_distance")
            agedb_acc = float(np.mean(agedb_accs) * 100)
            agedb_t = float(np.mean(agedb_thr))
            pbar.write(f"Verification done (AgeDB-30): acc={agedb_acc:.2f}%, t={agedb_t:.4f}")

            # Track last LFW metrics in epoch summary
            last_verif_acc, last_verif_t = lfw_acc, lfw_t

            pbar.disable = False
            pbar.refresh()

            # verify_10fold switches the model to eval mode; restore training
            model.train()
            head.train()

        # live update
        pbar.set_postfix(
            loss=running_loss / max(1, pbar.n),
            acc=correct / max(1, total),
        )

    train_acc = correct / max(total, 1)
    train_loss = running_loss / max(len(train_loader), 1)

    # --- ETA ---
    epoch_time = time.time() - epoch_start
    elapsed = time.time() - start_time
    remaining = (num_epochs - (epoch + 1)) * epoch_time
    eta = time.strftime("%H:%M:%S", time.gmtime(remaining))

    verif_acc_str = f"{last_verif_acc:.4f}" if last_verif_acc is not None else "n/a"
    verif_t_str = f"{last_verif_t:.2f}" if last_verif_t is not None else "n/a"

    print(
        f"epoch {epoch+1}/{num_epochs} "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.6f} "
        f"verif_acc={verif_acc_str}% verif_t={verif_t_str} "
        f"epoch_time={epoch_time:.1f}s ETA={eta}"
    )

    scheduler.step()

    # --- checkpoint ---
    ckpt_path = ckpt_dir / f"epoch_{epoch+1}.pt"
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "head_state": head.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": train_loss,
        },
        ckpt_path,
    )

epoch 1/20:   0%|          | 0/11373 [00:00<?, ?it/s]

Running verification at step 2000...
Verification done (LFW): acc=66.18%, t=1.7662
Verification done (CFP-FP): acc=61.20%, t=1.9224


epoch 1/20:  18%|█▊        | 2002/11373 [03:23<4:45:20,  1.83s/it, acc=9.66e-5, loss=42.7]

Verification done (AgeDB-30): acc=52.52%, t=1.7245


Running verification at step 4000...
Verification done (LFW): acc=66.28%, t=1.8293
Verification done (CFP-FP): acc=62.14%, t=2.0086


...

Verification done (AgeDB-30): acc=59.78%, t=1.9848


Running verification at step 124000...
Verification done (LFW): acc=79.78%, t=1.6334
Verification done (CFP-FP): acc=64.54%, t=2.0029


epoch 11/20:  90%|█████████ | 10271/11373 [29:50<1:01:54,  3.37s/it, acc=0.0394, loss=22.7]

Verification done (AgeDB-30): acc=60.27%, t=1.9847


epoch 11/20:  98%|█████████▊| 11116/11373 [32:32<00:52,  4.86it/s, acc=0.0395, loss=22.7]  

In [ ]:
# load a checkpoint and run verification sweep
ckpt_path = "checkpoints/epoch_14.pt"  # replace with your .pth/.pt path
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state["model_state"])

pairs = build_pairs_from_index(
    index_file=str(Path("~/Datasets").expanduser() / "CASIA" / "index.txt"),
    num_pairs=2000
)

last_verif_acc, last_verif_t, _ = verify_10fold(
    model,
    transform,
    pairs_lfw,
    device,
    flip=True,
    thresholds=np.linspace(0, 4, 401),
    seed=123
)

print("best verification acc:", best_acc, "best threshold:", best_t)
